# UX Metrics Processing Notebook

This notebook implements the full metrics process end-to-end, including explicit handling of repeated `tot_ms` entries.

## Important note on "how it was handled before"
In the earlier thesis analysis, repeated `tot_ms` values were aggregated using **mean per participant and `ui_condition`** to produce one ToT value per condition.

In this notebook, Section 3 implements a more generic event-log rule: **sort by timestamp and keep the latest record** for each duplicate key, while also reporting alternate aggregates (`max`, `sum`) for audit.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)
sns.set_theme(style='whitegrid')

## 1) Load Raw Metrics Files
Load raw metrics from CSV/XLSX, then combine into a single metrics table with source metadata.

In [ ]:
# Paths
DATA_DIR = Path('/Users/gvnd/Documents/College/Skripsian/data')
TOT_PATH = DATA_DIR / 'tot_results.csv'
QUESTIONNAIRE_PATH = DATA_DIR / 'Usability Testing Aplikasi Mindscape (Responses) (1).xlsx'

# Raw loads
raw_tot = pd.read_csv(TOT_PATH)
raw_quest = pd.read_excel(QUESTIONNAIRE_PATH)

# Add source metadata for traceability
raw_tot['source_file'] = TOT_PATH.name
raw_quest['source_file'] = QUESTIONNAIRE_PATH.name

print('raw_tot shape:', raw_tot.shape)
print('raw_quest shape:', raw_quest.shape)
raw_tot.head()

## 2) Standardize Schema and Data Types
Normalize names, parse timestamps, cast numeric columns, and enforce required fields.

In [ ]:
tot = raw_tot.copy()
quest = raw_quest.copy()

# Normalize key names
for df in (tot, quest):
    if 'Nama Tester' in df.columns:
        df['nama_tester_clean'] = df['Nama Tester'].astype(str).str.strip().str.lower()

# Parse and cast
required_tot_cols = ['Nama Tester', 'ui_condition', 'tot_ms', 'logged_at']
missing_tot = [c for c in required_tot_cols if c not in tot.columns]
assert not missing_tot, f'Missing required ToT columns: {missing_tot}'

tot['tot_ms'] = pd.to_numeric(tot['tot_ms'], errors='coerce')
tot['tot_sec'] = tot['tot_ms'] / 1000.0
tot['logged_at'] = pd.to_datetime(tot['logged_at'], errors='coerce')

eri_cols = list(quest.columns[4:12])
pei_cols = list(quest.columns[12:20])
for c in eri_cols + pei_cols:
    quest[c] = pd.to_numeric(quest[c], errors='coerce')

quest['ERI_Composite'] = quest[eri_cols].mean(axis=1)
quest['PEI_Composite'] = quest[pei_cols].mean(axis=1)

# Group mapping
quest['Group'] = quest['Group'].astype(str).str.strip().replace({'nan': pd.NA})
device_map = {'A': 'iPhone 17', 'B': 'iPhone 17', 'C': 'Mi Pad 5', 'D': 'Mi Pad 5'}
order_map = {'A': 'Standard First', 'B': 'Rush Hour First', 'C': 'Standard First', 'D': 'Rush Hour First'}
quest['Device'] = quest['Group'].map(device_map)
quest['UI_Order'] = quest['Group'].map(order_map)

print('ToT null tot_ms:', tot['tot_ms'].isna().sum())
print('ToT null logged_at:', tot['logged_at'].isna().sum())
print('Questionnaire shape:', quest.shape)

## 3) Deduplicate/Resolve Multiple `tot_ms` Entries
Deterministic rule for event-style logs: group by stable keys and keep the latest timestamped record. Also compute duplicate counts and alternate audit aggregates.

For this dataset, stable keys are approximated by:
- `nama_tester_clean`
- `ui_condition`

Because this dataset is per interaction, this strategy selects the latest recorded interaction per participant and UI condition.

Note: Earlier thesis runs used **mean per participant x UI condition**; both outputs are produced below for transparency.

In [ ]:
stable_keys = ['nama_tester_clean', 'ui_condition']

# Duplicate incidence
dup_counts = (
    tot.groupby(stable_keys, dropna=False)
       .size()
       .rename('n_rows')
       .reset_index()
)
dup_counts['is_duplicate'] = dup_counts['n_rows'] > 1

# Deterministic latest-record resolution
latest_tot = (
    tot.sort_values('logged_at')
       .groupby(stable_keys, as_index=False)
       .tail(1)
       .copy()
)

# Alternate aggregates for audit
audit_aggs = (
    tot.groupby(stable_keys, as_index=False)
       .agg(
           tot_sec_mean=('tot_sec', 'mean'),
           tot_sec_max=('tot_sec', 'max'),
           tot_sec_sum=('tot_sec', 'sum'),
           n_records=('tot_sec', 'size'),
       )
)

# Earlier thesis-compatible aggregation (mean per participant x condition)
mean_tot = (
    tot.groupby(stable_keys, as_index=False)
       .agg(tot_sec=('tot_sec', 'mean'))
)

print('Total key groups:', len(dup_counts))
print('Groups with duplicates:', int(dup_counts['is_duplicate'].sum()))
display(dup_counts[dup_counts['is_duplicate']].sort_values('n_rows', ascending=False).head(10))
display(audit_aggs.head())

## 4) Compute Core Metrics (Latency, Throughput, Error Rate)
Compute per-event latency from resolved `tot_ms`, throughput over fixed time windows, and error rate from status-like fields where available.

In [ ]:
# Use latest-record resolved data for generic event-log KPI computation
resolved_events = latest_tot.copy()
resolved_events['latency_ms'] = resolved_events['tot_ms']
resolved_events['latency_sec'] = resolved_events['tot_sec']

# Throughput per 1-minute window
throughput = (
    resolved_events
    .set_index('logged_at')
    .groupby(pd.Grouper(freq='1min'))
    .size()
    .rename('events_per_min')
    .reset_index()
)

# Error-rate placeholder: this dataset has no explicit status column.
status_col = None
for c in ['status', 'result', 'is_error', 'success']:
    if c in resolved_events.columns:
        status_col = c
        break

if status_col is None:
    error_rate = np.nan
    print('No explicit status/result field found; error rate set to NaN for this dataset.')
else:
    series = resolved_events[status_col].astype(str).str.lower()
    error_rate = series.str.contains('error|fail|false').mean()

print('Resolved events:', len(resolved_events))
print('Mean latency (sec):', round(resolved_events['latency_sec'].mean(), 3))
print('Error rate:', error_rate)
display(throughput.head())

## 5) Aggregate by Run, Endpoint, and Time Window
Build grouped summaries with central tendency and percentile statistics for comparison across executions.

This dataset does not include explicit `run_id` or `endpoint`, so those dimensions are proxied by:
- `source_file` as run identifier
- `ui_condition` as endpoint-like category

In [ ]:
def p50(x): return np.percentile(x, 50)
def p90(x): return np.percentile(x, 90)
def p95(x): return np.percentile(x, 95)
def p99(x): return np.percentile(x, 99)

resolved_events['run_id'] = resolved_events['source_file']
resolved_events['endpoint'] = resolved_events['ui_condition']
resolved_events['time_bucket_5min'] = resolved_events['logged_at'].dt.floor('5min')

agg = (
    resolved_events
    .groupby(['run_id', 'endpoint', 'time_bucket_5min'], dropna=False)
    .agg(
        n=('latency_sec', 'size'),
        mean=('latency_sec', 'mean'),
        p50=('latency_sec', p50),
        p90=('latency_sec', p90),
        p95=('latency_sec', p95),
        p99=('latency_sec', p99),
        min=('latency_sec', 'min'),
        max=('latency_sec', 'max'),
        std=('latency_sec', 'std'),
    )
    .reset_index()
)

display(agg.head())

## 6) Add Data Quality Checks and Assertions
Validate schema and values: missing keys, negative durations, duplicate-key counts, and outlier detection.

In [ ]:
validation = {
    'missing_nama_tester': int(tot['nama_tester_clean'].isna().sum()),
    'missing_ui_condition': int(tot['ui_condition'].isna().sum()),
    'missing_logged_at': int(tot['logged_at'].isna().sum()),
    'negative_tot_ms': int((tot['tot_ms'] < 0).sum()),
    'duplicate_key_groups': int((dup_counts['n_rows'] > 1).sum()),
}

# Outlier rule: above Q3 + 1.5*IQR on resolved latency
q1, q3 = resolved_events['latency_sec'].quantile([0.25, 0.75])
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
validation['outlier_count_iqr'] = int((resolved_events['latency_sec'] > upper_bound).sum())

validation_df = pd.DataFrame({'check': list(validation.keys()), 'value': list(validation.values())})
display(validation_df)

assert validation['negative_tot_ms'] == 0, 'Negative tot_ms detected.'
assert validation['missing_ui_condition'] == 0, 'Missing ui_condition detected.'

## 7) Visualize Metric Distributions and Trends
Plot latency distribution, trend over time, and duplicate incidence.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.histplot(resolved_events['latency_sec'], kde=True, ax=axes[0], color='#4e79a7')
axes[0].set_title('Latency Distribution (sec)')

sns.boxplot(data=resolved_events, x='ui_condition', y='latency_sec', ax=axes[1], palette='Set2')
axes[1].set_title('Latency by UI Condition')
axes[1].set_xlabel('ui_condition')

dup_per_ui = dup_counts.groupby('ui_condition', as_index=False)['is_duplicate'].sum()
sns.barplot(data=dup_per_ui, x='ui_condition', y='is_duplicate', ax=axes[2], color='#f28e2b')
axes[2].set_title('Duplicate Key Groups by UI Condition')
axes[2].set_ylabel('duplicate groups')

plt.tight_layout()
plt.show()

trend = agg.groupby('time_bucket_5min', as_index=False)['p95'].mean()
plt.figure(figsize=(10, 4))
plt.plot(trend['time_bucket_5min'], trend['p95'], marker='o')
plt.title('P95 Latency Trend (5-min buckets)')
plt.xlabel('time')
plt.ylabel('p95 latency (sec)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8) Export Clean Metrics and Summary Tables
Write resolved event-level data, KPI aggregates, and validation report for downstream reuse.

In [ ]:
OUT_DIR = Path('analysis_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

resolved_events.to_csv(OUT_DIR / 'resolved_events_latest.csv', index=False)
audit_aggs.to_csv(OUT_DIR / 'tot_duplicate_audit.csv', index=False)
agg.to_csv(OUT_DIR / 'kpi_aggregates.csv', index=False)
validation_df.to_csv(OUT_DIR / 'validation_report.csv', index=False)

print('Exported files:')
for p in sorted(OUT_DIR.glob('*')):
    print('-', p)

## Thesis-Specific Statistical Workflow (Your Chapter 4 Process)
This section reproduces the usability analysis pipeline (descriptive stats, reliability, normality, paired test, correlations, and confound checks) with a configurable ToT resolution strategy.

Set `TOT_STRATEGY` to:
- `"mean"` to replicate prior runs in this chat (mean per participant x UI condition)
- `"latest"` to use latest timestamped record per participant x UI condition
- `"earliest"` to use earliest timestamped record per participant x UI condition (recommended to reduce learning-effect contamination)

In [ ]:
TOT_STRATEGY = 'earliest'  # options: 'mean', 'latest', 'earliest'

if TOT_STRATEGY == 'mean':
    tot_for_thesis = (
        tot.groupby(['nama_tester_clean', 'ui_condition'], as_index=False)
           .agg(tot_sec=('tot_sec', 'mean'))
    )
elif TOT_STRATEGY == 'latest':
    tot_for_thesis = (
        tot.sort_values('logged_at')
           .groupby(['nama_tester_clean', 'ui_condition'], as_index=False)
           .tail(1)[['nama_tester_clean', 'ui_condition', 'tot_sec']]
    )
elif TOT_STRATEGY == 'earliest':
    tot_for_thesis = (
        tot.sort_values('logged_at')
           .groupby(['nama_tester_clean', 'ui_condition'], as_index=False)
           .head(1)[['nama_tester_clean', 'ui_condition', 'tot_sec']]
    )
else:
    raise ValueError('TOT_STRATEGY must be mean, latest, or earliest')

pivot = (
    tot_for_thesis.pivot_table(
        index='nama_tester_clean',
        columns='ui_condition',
        values='tot_sec',
        aggfunc='mean'
    )
    .reset_index()
    .rename(columns={'standard_ui': 'ToT_Standard', 'rush_hour_ui': 'ToT_RushHour'})
)

q_keep = quest[['nama_tester_clean', 'Nama Tester', 'Group', 'Device', 'UI_Order', 'ERI_Composite', 'PEI_Composite'] + eri_cols + pei_cols]
merged = pd.merge(q_keep, pivot, on='nama_tester_clean', how='inner')
paired = merged.dropna(subset=['ToT_Standard', 'ToT_RushHour']).copy()
paired['ToT_Savings'] = paired['ToT_Standard'] - paired['ToT_RushHour']

print('TOT strategy:', TOT_STRATEGY)
print('N paired:', len(paired))

# Assumption + tests
def cronbach_alpha(df_items):
    X = df_items.dropna().to_numpy(float)
    k = X.shape[1]
    return (k / (k - 1)) * (1 - (X.var(axis=0, ddof=1).sum() / X.sum(axis=1).var(ddof=1)))

alpha_eri = cronbach_alpha(paired[eri_cols])
alpha_pei = cronbach_alpha(paired[pei_cols])
sh_w, sh_p = stats.shapiro(paired['ToT_Savings'])

if sh_p > 0.05:
    test_name = 'Paired t-test'
    stat, pval = stats.ttest_rel(paired['ToT_Standard'], paired['ToT_RushHour'])
    stat_label = 't'
else:
    test_name = 'Wilcoxon signed-rank'
    stat, pval = stats.wilcoxon(paired['ToT_Standard'], paired['ToT_RushHour'])
    stat_label = 'W'

cohen_dz = paired['ToT_Savings'].mean() / paired['ToT_Savings'].std(ddof=1)

# Correlations
norm_x = stats.shapiro(paired['ToT_Savings']).pvalue > 0.05
norm_y = stats.shapiro(paired['ERI_Composite']).pvalue > 0.05
if norm_x and norm_y:
    c1_method = 'Pearson'
    c1_r, c1_p = stats.pearsonr(paired['ToT_Savings'], paired['ERI_Composite'])
else:
    c1_method = 'Spearman'
    c1_r, c1_p = stats.spearmanr(paired['ToT_Savings'], paired['ERI_Composite'])

norm_a = stats.shapiro(paired['ERI_Composite']).pvalue > 0.05
norm_b = stats.shapiro(paired['PEI_Composite']).pvalue > 0.05
if norm_a and norm_b:
    c2_method = 'Pearson'
    c2_r, c2_p = stats.pearsonr(paired['ERI_Composite'], paired['PEI_Composite'])
else:
    c2_method = 'Spearman'
    c2_r, c2_p = stats.spearmanr(paired['ERI_Composite'], paired['PEI_Composite'])

# Confounds
iphone = paired.loc[paired['Device'] == 'iPhone 17', 'ToT_RushHour']
mipad = paired.loc[paired['Device'] == 'Mi Pad 5', 'ToT_RushHour']
std_first = paired.loc[paired['UI_Order'] == 'Standard First', 'ToT_RushHour']
rush_first = paired.loc[paired['UI_Order'] == 'Rush Hour First', 'ToT_RushHour']

t_dev, p_dev = stats.ttest_ind(iphone, mipad, equal_var=False)
t_ord, p_ord = stats.ttest_ind(std_first, rush_first, equal_var=False)

# APA-style tables
desc = pd.DataFrame([
    ('ToT_Standard (s)', len(paired), paired['ToT_Standard'].mean(), paired['ToT_Standard'].std(ddof=1)),
    ('ToT_RushHour (s)', len(paired), paired['ToT_RushHour'].mean(), paired['ToT_RushHour'].std(ddof=1)),
    ('ERI Composite (1-5)', len(paired), paired['ERI_Composite'].mean(), paired['ERI_Composite'].std(ddof=1)),
    ('PEI Composite (1-5)', len(paired), paired['PEI_Composite'].mean(), paired['PEI_Composite'].std(ddof=1)),
], columns=['Measure', 'N', 'M', 'SD'])

assump = pd.DataFrame([
    ('Cronbach alpha ERI (8 items)', alpha_eri, np.nan),
    ('Cronbach alpha PEI (8 items)', alpha_pei, np.nan),
    ('Shapiro-Wilk (ToT difference)', sh_w, sh_p),
], columns=['Test', 'Statistic', 'p'])

primary = pd.DataFrame([
    (test_name, stat_label, stat, pval, cohen_dz, paired['ToT_Savings'].mean())
], columns=['Test', 'StatLabel', 'Statistic', 'p', "Cohen_dz", 'Mean_ToT_Savings'])

corrs = pd.DataFrame([
    ('ToT Savings vs ERI', c1_method, c1_r, c1_p),
    ('ERI vs PEI', c2_method, c2_r, c2_p),
], columns=['Variables', 'Method', 'r', 'p'])

confounds = pd.DataFrame([
    ('Device: iPhone 17 vs Mi Pad 5 (RushHour ToT)', t_dev, p_dev, len(iphone), len(mipad)),
    ('Order: Standard First vs Rush Hour First (RushHour ToT)', t_ord, p_ord, len(std_first), len(rush_first)),
], columns=['Comparison', 't', 'p', 'n_group1', 'n_group2'])

print('\nStep 1: Descriptive Statistics')
display(desc.round(3))
print('\nStep 2: Assumptions')
display(assump.round(4))
print('\nStep 3: Primary Hypothesis Test')
display(primary.round(4))
print('\nStep 4: Correlations')
display(corrs.round(4))
print('\nStep 5: Confound Checks')
display(confounds.round(4))

## Reproducibility Notes
- Previous thesis analysis in this chat used: **mean aggregation** for repeated `tot_ms` per participant x `ui_condition`.
- Generic event-log best practice shown in this notebook: **latest timestamp wins** for duplicate stable keys.
- To reduce learning effect in within-subject flows, this notebook now supports **earliest timestamp wins** and sets it as the default in the thesis cell.
- Audit outputs include alternate aggregates (`mean`, `max`, `sum`) to compare sensitivity.

### Optional alternatives
```python
# Latest record per participant x condition
latest_tot = (
    tot.sort_values('logged_at')
       .groupby(['nama_tester_clean', 'ui_condition'], as_index=False)
       .tail(1)
)

# Earliest record per participant x condition
earliest_tot = (
    tot.sort_values('logged_at')
       .groupby(['nama_tester_clean', 'ui_condition'], as_index=False)
       .head(1)
)
```